Prepare your training and validation data:

In [1]:
import numpy as np
import pandas as pd

# 分割数据为特征（X）和标签（y）
df_zh2ko = pd.read_csv("yulgang global script_zh-CN_ko.csv")
df_zh2ko.replace(np.nan, None)
x_train_zh2ko = df_zh2ko["zh-CN"].to_list()  # 替换'标签列名称'为你实际的标签列名称
y_train_zh2ko = df_zh2ko['ko'].to_list()  # 替换'标签列名称'为你实际的标签列名称val

In [2]:
import numpy as np
import pandas as pd

# 分割数据为特征（X）和标签（y）
df_zh2en = pd.read_csv("yulgang global script_zh-CN_en.csv")
df_zh2en.replace(np.nan, None)
x_train_zh2en = df_zh2en["zh-CN"].to_list()  # 替换'标签列名称'为你实际的标签列名称
y_train_zh2en = df_zh2en['en'].to_list()  # 替换'标签列名称'为你实际的标签列名称val

In [3]:
import numpy as np
import pandas as pd

# 分割数据为特征（X）和标签（y）
df_zh2th = pd.read_csv("yulgang global script_zh-CN_th.csv")
df_zh2th.replace(np.nan, None)
x_train_zh2th = df_zh2th["zh-CN"].to_list()  # 替换'标签列名称'为你实际的标签列名称
y_train_zh2th = df_zh2th['th'].to_list()  # 替换'标签列名称'为你实际的标签列名称val

Load the tokenizer:

In [4]:
from transformers import AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, tgt_lang=None)

In [5]:
# add tokens not in the vocabulary of the tokenizer
chars = list(
    set(''.join(x_train_zh2ko + y_train_zh2ko + x_train_zh2en + y_train_zh2en + x_train_zh2th + y_train_zh2th)))
chars_not_in_vocab = [char for char in chars if 3 in tokenizer(char).input_ids]  # 3 is the value of unknown words
tokenizer.add_tokens(chars_not_in_vocab)

1447

In [6]:
# Note: we're now creating separate encodings for the inputs and outputs.
# truncation: truncate the sequence to a shorter length, because sometimes a sequence may be too long for a model to handle
# padding: Padding is a strategy for ensuring tensors are rectangular by adding a special padding token to shorter sentences.
# return_tensors: If set 'pt', will return tensors instead of list of python integers. Acceptable values are PyTorch torch.Tensor objects.
tokenizer.src_lang = "zho_Hans"
tokenizer.tgt_lang = "kor_Hang"
train_encodings_zh2ko = tokenizer(x_train_zh2ko, text_target=y_train_zh2ko, truncation=True, padding=True,
                                  return_tensors="pt")
tokenizer.tgt_lang = "eng_Latn"
train_encodings_zh2en = tokenizer(x_train_zh2en, text_target=y_train_zh2en, truncation=True, padding=True,
                                  return_tensors="pt")
tokenizer.tgt_lang = "tha_Thai"
train_encodings_zh2th = tokenizer(x_train_zh2th, text_target=y_train_zh2th, truncation=True, padding=True,
                                  return_tensors="pt")

Convert your encodings into torch Datasets object:

In [7]:
import torch
from transformers import BatchEncoding


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data_encoded_list: list):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []
        for data_encoded in data_encoded_list:
            self.input_ids.extend(data_encoded.data["input_ids"])
            self.attention_mask.extend(data_encoded.data["attention_mask"])
            self.labels.extend(data_encoded.data["labels"])

    def __getitem__(self, index):
        item = {"input_ids": self.input_ids[index],
                "attention_mask": self.attention_mask[index],
                "labels": self.labels[index]}
        return item

    def __len__(self):
        return len(self.input_ids)


train_dataset = TranslationDataset([train_encodings_zh2ko, train_encodings_zh2en, train_encodings_zh2th])

Load the pretrained model:

In [13]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [14]:
# 调整embedding层的大小
model.resize_token_embeddings(len(tokenizer))

Embedding(257651, 1024)

Define your training arguments and train the model:

In [15]:
custom_model_name = "yulgang_zh2enkoth_0829"

In [16]:
from transformers import Trainer, TrainingArguments, IntervalStrategy

# fp16：半精度运算，启用后提高一倍以上运算速度，不影响loss
# gradient_accumulation_steps：steps越大，速度越快，loss越高
# gradient_checkpointing：启用后，降低30%左右速度，节省显存2/3
# per_device_train_batch_size：size越大，GPU占用率越大，速度越快，loss越高，几乎成正比
training_args = TrainingArguments(custom_model_name,
                                  num_train_epochs=1,
                                  per_device_eval_batch_size=1,
                                  per_device_train_batch_size=1,
                                  gradient_accumulation_steps=1,
                                  gradient_checkpointing=True,
                                  fp16=True,
                                  logging_strategy=IntervalStrategy.STEPS,
                                  logging_steps=1000,
                                  save_strategy=IntervalStrategy.STEPS,
                                  save_steps=1000,
                                  save_total_limit=1,
                                  )
from torch.utils import checkpoint  #未知的bug：不会自动加载这个包

In [17]:
trainer = Trainer(
    model=model,  # the instantiated 🤗 Transformers model to be trained
    args=training_args,  # training arguments, defined above
    train_dataset=train_dataset,  # training dataset
)

trainer.train(resume_from_checkpoint=False)

/root/miniconda3/lib/python3.8/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
1000,1.395600
2000,0.047800
3000,0.049500
4000,0.042000
5000,0.039900
6000,0.042900
7000,0.039300
8000,0.040800
9000,0.041500
10000,0.036600


TrainOutput(global_step=177281, training_loss=0.03410114932067369, metrics={'train_runtime': 64811.5136, 'train_samples_per_second': 2.735, 'train_steps_per_second': 2.735, 'total_flos': 3.841864709670175e+17, 'train_loss': 0.03410114932067369, 'epoch': 1.0})

Save your fine-tuned model and tokenizer:

In [18]:
trainer.save_model(custom_model_name)
tokenizer.save_pretrained(custom_model_name)

('yulgang_zh2enkoth_0829/tokenizer_config.json',
 'yulgang_zh2enkoth_0829/special_tokens_map.json',
 'yulgang_zh2enkoth_0829/tokenizer.json')